# Лабораторна робота №3
## Transfer Learning у комп'ютерному зорі

**Мета роботи:** Опанувати стратегії роботи з моделями глибинного навчання для задач класифікації зображень, візуалізувати процес навчання в TensorBoard.

**Датасет:** Oxford 102 Flower Categories  
**Моделі:** ResNet50 та EfficientNet-B0  
**Стратегії:** Feature Extraction → Fine-Tuning

In [1]:
# Встановлення залежностей (розкоментуйте за необхідності)
# !pip install torch torchvision tensorboard matplotlib seaborn scikit-learn pandas numpy Pillow grad-cam

## 1. Імпорти та конфігурація

In [2]:
import os
import copy
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
from torch.utils.tensorboard import SummaryWriter

import torchvision
from torchvision import models, datasets, transforms
from torchvision.transforms import v2

from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image

warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = Path('./data')
LOG_DIR  = Path('./runs')
CKPT_DIR = Path('./checkpoints')

for d in [DATA_DIR, LOG_DIR, CKPT_DIR]:
    d.mkdir(exist_ok=True)

print(f'Device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')
print(f'Torchvision version: {torchvision.__version__}')

ModuleNotFoundError: No module named 'tensorboard'

## 2. Завантаження датасету

Використовуємо **Oxford 102 Flower Categories** — стандартний бенчмарк для transfer learning.  
Датасет містить 8189 зображень квіток, розподілених по 102 класах.

In [ ]:
# Базові трансформації для завантаження (без аугментацій)
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Завантаження датасету
full_train = datasets.Flowers102(root=DATA_DIR, split='train', download=True, transform=base_transform)
full_val   = datasets.Flowers102(root=DATA_DIR, split='val',   download=True, transform=base_transform)
full_test  = datasets.Flowers102(root=DATA_DIR, split='test',  download=True, transform=base_transform)

NUM_CLASSES = 102

print(f'Train samples : {len(full_train)}')
print(f'Val samples   : {len(full_val)}')
print(f'Test samples  : {len(full_test)}')
print(f'Total classes : {NUM_CLASSES}')

## 3. Розвідувальний аналіз даних (EDA)

In [ ]:
# Розподіл класів у тренувальному наборі
train_labels = [full_train[i][1] for i in range(len(full_train))]
label_counts  = Counter(train_labels)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Гістограма кількості зображень по класах
counts_sorted = sorted(label_counts.values())
axes[0].hist(counts_sorted, bins=20, color='steelblue', edgecolor='black')
axes[0].set_title('Розподіл кількості зображень по класах', fontsize=13)
axes[0].set_xlabel('Кількість зображень')
axes[0].set_ylabel('Кількість класів')
axes[0].axvline(np.mean(counts_sorted), color='red', linestyle='--', label=f'Середнє: {np.mean(counts_sorted):.1f}')
axes[0].legend()

# Top-20 класів
top20 = sorted(label_counts.items(), key=lambda x: x[1], reverse=True)[:20]
axes[1].barh([f'Клас {c}' for c, _ in top20], [n for _, n in top20], color='coral')
axes[1].set_title('Топ-20 класів за кількістю зображень', fontsize=13)
axes[1].set_xlabel('Кількість зображень')

plt.tight_layout()
plt.savefig('eda_class_distribution.png', dpi=120)
plt.show()

print(f'Мін. зображень у класі : {min(counts_sorted)}')
print(f'Макс. зображень у класі: {max(counts_sorted)}')
print(f'Середнє по класу       : {np.mean(counts_sorted):.1f}')

In [ ]:
# Перегляд розмірів зображень
sizes = []
for i in range(min(200, len(full_train))):
    img, _ = full_train[i]
    sizes.append((img.shape[1], img.shape[2]))

heights, widths = zip(*sizes)
print(f'Висоти після resize: {set(heights)}')
print(f'Ширини після resize: {set(widths)}')

# Відображення прикладів з кожного спліту
fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for i, ax in enumerate(axes[0]):
    img_tensor, label = full_train[i * 10]
    ax.imshow(img_tensor.permute(1, 2, 0).numpy())
    ax.set_title(f'Клас {label}', fontsize=8)
    ax.axis('off')
for i, ax in enumerate(axes[1]):
    img_tensor, label = full_train[i * 10 + 5]
    ax.imshow(img_tensor.permute(1, 2, 0).numpy())
    ax.set_title(f'Клас {label}', fontsize=8)
    ax.axis('off')
axes[0][0].set_ylabel('Train', fontsize=10)
plt.suptitle('Приклади зображень з тренувального набору', fontsize=13)
plt.tight_layout()
plt.savefig('eda_samples.png', dpi=120)
plt.show()

## 4. Попередня обробка та аугментація

Застосовуємо:
- **RandomResizedCrop** — вирізаємо випадковий патч, щоб модель бачила різні масштаби
- **RandomHorizontalFlip** — горизонтальне відображення для інваріантності
- **ColorJitter** — варіації яскравості/контрасту для робастності
- **Normalize** — нормалізація за статистиками ImageNet (оскільки base-моделі тренувались на ньому)

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Перезавантажуємо датасети з правильними трансформаціями
train_ds = datasets.Flowers102(root=DATA_DIR, split='train', download=False, transform=train_transform)
val_ds   = datasets.Flowers102(root=DATA_DIR, split='val',   download=False, transform=val_transform)
test_ds  = datasets.Flowers102(root=DATA_DIR, split='test',  download=False, transform=val_transform)

BATCH_SIZE = 32

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches  : {len(val_loader)}')
print(f'Test batches : {len(test_loader)}')

# Демонстрація аугментацій
fig, axes = plt.subplots(2, 6, figsize=(18, 7))
original_transform = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor()])
orig_ds = datasets.Flowers102(root=DATA_DIR, split='train', download=False, transform=original_transform)

for col in range(6):
    orig_img, label = orig_ds[col * 15]
    aug_img,  _     = train_ds[col * 15]
    axes[0][col].imshow(orig_img.permute(1,2,0).numpy())
    axes[0][col].set_title(f'Оригінал (кл.{label})', fontsize=8)
    axes[0][col].axis('off')
    # Денормалізуємо для відображення
    disp = aug_img.clone()
    for c, (m, s) in enumerate(zip(IMAGENET_MEAN, IMAGENET_STD)):
        disp[c] = disp[c] * s + m
    disp = disp.clamp(0, 1).permute(1, 2, 0).numpy()
    axes[1][col].imshow(disp)
    axes[1][col].set_title('Аугментовано', fontsize=8)
    axes[1][col].axis('off')

axes[0][0].set_ylabel('Оригінал', fontsize=9)
axes[1][0].set_ylabel('Аугментація', fontsize=9)
plt.suptitle('Порівняння оригінальних та аугментованих зображень', fontsize=13)
plt.tight_layout()
plt.savefig('augmentation_comparison.png', dpi=120)
plt.show()

## 5. Допоміжні функції для тренування

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, scaler=None):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        if scaler is not None:
            with torch.amp.autocast('cuda'):
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        _, preds = outputs.max(1)
        correct  += preds.eq(labels).sum().item()
        total    += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * imgs.size(0)
        _, preds = outputs.max(1)
        correct  += preds.eq(labels).sum().item()
        total    += imgs.size(0)
    return total_loss / total, correct / total


def train_model(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    device, writer_tag, num_epochs=15, patience=5, ckpt_name='best.pth'
):
    """
    Повний цикл тренування з early stopping, LR scheduling та TensorBoard.
    """
    writer = SummaryWriter(log_dir=str(LOG_DIR / writer_tag))
    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

    best_val_acc = 0.0
    no_improve   = 0
    history      = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler)
        val_loss,   val_acc   = evaluate(model, val_loader, criterion, device)

        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()

        current_lr = optimizer.param_groups[0]['lr']

        # TensorBoard логування
        writer.add_scalars('Loss',     {'train': train_loss, 'val': val_loss},   epoch)
        writer.add_scalars('Accuracy', {'train': train_acc,  'val': val_acc},    epoch)
        writer.add_scalar('LearningRate', current_lr, epoch)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        elapsed = time.time() - t0
        print(f'Epoch [{epoch:02d}/{num_epochs}] '
              f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
              f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | '
              f'LR: {current_lr:.2e} | Time: {elapsed:.1f}s')

        # Зберігаємо найкращу модель
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            no_improve   = 0
            torch.save(model.state_dict(), CKPT_DIR / ckpt_name)
            print(f'  ✓ Збережено нові найкращі ваги (val_acc={val_acc:.4f})')
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stopping на епосі {epoch}')
                break

    writer.close()
    print(f'\nНайкраща val_acc: {best_val_acc:.4f}')
    return history, best_val_acc


def plot_history(history, title='Training History', save_name='history.png'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(history['train_loss']) + 1)

    ax1.plot(epochs, history['train_loss'], 'b-o', label='Train')
    ax1.plot(epochs, history['val_loss'],   'r-o', label='Val')
    ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True)

    ax2.plot(epochs, history['train_acc'],  'b-o', label='Train')
    ax2.plot(epochs, history['val_acc'],    'r-o', label='Val')
    ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True)

    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(save_name, dpi=120)
    plt.show()

---
## 6. Модель 1: ResNet50
### 6.1 Фаза 1 — Feature Extraction (заморожена база + новий head)

In [ ]:
def build_resnet50(num_classes, freeze_base=True):
    weights = models.ResNet50_Weights.IMAGENET1K_V2
    model   = models.resnet50(weights=weights)

    if freeze_base:
        for param in model.parameters():
            param.requires_grad = False

    # Замінюємо класифікаційний head
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(512, num_classes),
    )
    return model


rn50 = build_resnet50(NUM_CLASSES, freeze_base=True).to(DEVICE)

trainable  = sum(p.numel() for p in rn50.parameters() if p.requires_grad)
total      = sum(p.numel() for p in rn50.parameters())
print(f'ResNet50 | Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Тренуємо лише head
optimizer_rn50_p1 = optim.AdamW(
    filter(lambda p: p.requires_grad, rn50.parameters()),
    lr=1e-3, weight_decay=1e-4
)
scheduler_rn50_p1 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_rn50_p1, mode='min', factor=0.5, patience=3
)

print('=== ResNet50 — Фаза 1: Feature Extraction ===')
hist_rn50_p1, best_rn50_p1 = train_model(
    rn50, train_loader, val_loader,
    criterion, optimizer_rn50_p1, scheduler_rn50_p1,
    DEVICE, writer_tag='resnet50/phase1',
    num_epochs=15, patience=5,
    ckpt_name='resnet50_phase1_best.pth'
)

plot_history(hist_rn50_p1, title='ResNet50 — Фаза 1 (Feature Extraction)', save_name='resnet50_phase1.png')

### 6.2 Фаза 2 — Fine-Tuning (розморожуємо верхні блоки ResNet)

In [ ]:
# Завантажуємо найкращі ваги першої фази
rn50.load_state_dict(torch.load(CKPT_DIR / 'resnet50_phase1_best.pth', map_location=DEVICE))

# Розморожуємо layer4 та layer3 (специфічні для задачі ознаки)
for name, param in rn50.named_parameters():
    if any(blk in name for blk in ['layer4', 'layer3', 'fc']):
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = sum(p.numel() for p in rn50.parameters() if p.requires_grad)
total     = sum(p.numel() for p in rn50.parameters())
print(f'ResNet50 Fine-Tune | Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

optimizer_rn50_p2 = optim.AdamW([
    {'params': [p for n, p in rn50.named_parameters() if 'fc'     in n and p.requires_grad], 'lr': 5e-4},
    {'params': [p for n, p in rn50.named_parameters() if 'layer4' in n and p.requires_grad], 'lr': 1e-4},
    {'params': [p for n, p in rn50.named_parameters() if 'layer3' in n and p.requires_grad], 'lr': 5e-5},
], weight_decay=1e-4)

scheduler_rn50_p2 = optim.lr_scheduler.CosineAnnealingLR(optimizer_rn50_p2, T_max=20, eta_min=1e-6)

In [ ]:
print('=== ResNet50 — Фаза 2: Fine-Tuning ===')
hist_rn50_p2, best_rn50_p2 = train_model(
    rn50, train_loader, val_loader,
    criterion, optimizer_rn50_p2, scheduler_rn50_p2,
    DEVICE, writer_tag='resnet50/phase2',
    num_epochs=25, patience=7,
    ckpt_name='resnet50_phase2_best.pth'
)

plot_history(hist_rn50_p2, title='ResNet50 — Фаза 2 (Fine-Tuning)', save_name='resnet50_phase2.png')

In [ ]:
# Оцінка на тестовому наборі
rn50.load_state_dict(torch.load(CKPT_DIR / 'resnet50_phase2_best.pth', map_location=DEVICE))
test_loss_rn50, test_acc_rn50 = evaluate(rn50, test_loader, criterion, DEVICE)
print(f'ResNet50 Test Loss: {test_loss_rn50:.4f} | Test Acc: {test_acc_rn50:.4f} ({test_acc_rn50*100:.2f}%)')

---
## 7. Модель 2: EfficientNet-B0
### 7.1 Фаза 1 — Feature Extraction

In [ ]:
def build_efficientnet_b0(num_classes, freeze_base=True):
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
    model   = models.efficientnet_b0(weights=weights)

    if freeze_base:
        for param in model.parameters():
            param.requires_grad = False

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 512),
        nn.SiLU(),
        nn.Dropout(p=0.2),
        nn.Linear(512, num_classes),
    )
    return model


eff_b0 = build_efficientnet_b0(NUM_CLASSES, freeze_base=True).to(DEVICE)

trainable = sum(p.numel() for p in eff_b0.parameters() if p.requires_grad)
total     = sum(p.numel() for p in eff_b0.parameters())
print(f'EfficientNet-B0 | Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
optimizer_eff_p1 = optim.AdamW(
    filter(lambda p: p.requires_grad, eff_b0.parameters()),
    lr=1e-3, weight_decay=1e-4
)
scheduler_eff_p1 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_eff_p1, mode='min', factor=0.5, patience=3
)

print('=== EfficientNet-B0 — Фаза 1: Feature Extraction ===')
hist_eff_p1, best_eff_p1 = train_model(
    eff_b0, train_loader, val_loader,
    criterion, optimizer_eff_p1, scheduler_eff_p1,
    DEVICE, writer_tag='efficientnet_b0/phase1',
    num_epochs=15, patience=5,
    ckpt_name='efficientnet_b0_phase1_best.pth'
)

plot_history(hist_eff_p1, title='EfficientNet-B0 — Фаза 1 (Feature Extraction)', save_name='effnet_phase1.png')

### 7.2 Фаза 2 — Fine-Tuning (розморожуємо останні блоки EfficientNet)

In [ ]:
eff_b0.load_state_dict(torch.load(CKPT_DIR / 'efficientnet_b0_phase1_best.pth', map_location=DEVICE))

# Розморожуємо останні 3 блоки та classifier
for name, param in eff_b0.named_parameters():
    if any(blk in name for blk in ['features.7', 'features.6', 'features.5', 'classifier']):
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = sum(p.numel() for p in eff_b0.parameters() if p.requires_grad)
total     = sum(p.numel() for p in eff_b0.parameters())
print(f'EfficientNet-B0 Fine-Tune | Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

optimizer_eff_p2 = optim.AdamW([
    {'params': [p for n, p in eff_b0.named_parameters() if 'classifier' in n and p.requires_grad], 'lr': 5e-4},
    {'params': [p for n, p in eff_b0.named_parameters() if 'features.7' in n and p.requires_grad], 'lr': 1e-4},
    {'params': [p for n, p in eff_b0.named_parameters() if 'features.6' in n and p.requires_grad], 'lr': 5e-5},
    {'params': [p for n, p in eff_b0.named_parameters() if 'features.5' in n and p.requires_grad], 'lr': 2e-5},
], weight_decay=1e-4)

scheduler_eff_p2 = optim.lr_scheduler.CosineAnnealingLR(optimizer_eff_p2, T_max=20, eta_min=1e-7)

In [ ]:
print('=== EfficientNet-B0 — Фаза 2: Fine-Tuning ===')
hist_eff_p2, best_eff_p2 = train_model(
    eff_b0, train_loader, val_loader,
    criterion, optimizer_eff_p2, scheduler_eff_p2,
    DEVICE, writer_tag='efficientnet_b0/phase2',
    num_epochs=25, patience=7,
    ckpt_name='efficientnet_b0_phase2_best.pth'
)

plot_history(hist_eff_p2, title='EfficientNet-B0 — Фаза 2 (Fine-Tuning)', save_name='effnet_phase2.png')

In [ ]:
eff_b0.load_state_dict(torch.load(CKPT_DIR / 'efficientnet_b0_phase2_best.pth', map_location=DEVICE))
test_loss_eff, test_acc_eff = evaluate(eff_b0, test_loader, criterion, DEVICE)
print(f'EfficientNet-B0 Test Loss: {test_loss_eff:.4f} | Test Acc: {test_acc_eff:.4f} ({test_acc_eff*100:.2f}%)')

---
## 8. TensorBoard — порівняння фаз та моделей

Запустіть у терміналі:
```bash
tensorboard --logdir=runs --port=6006
```
Або виконайте клітинку нижче.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 9. Зведене порівняння двох моделей

In [ ]:
results = {
    'ResNet50 Phase1 (FE)':      {'val_acc': best_rn50_p1, 'test_acc': None},
    'ResNet50 Phase2 (FT)':      {'val_acc': best_rn50_p2, 'test_acc': test_acc_rn50},
    'EfficientNet-B0 Phase1 (FE)': {'val_acc': best_eff_p1, 'test_acc': None},
    'EfficientNet-B0 Phase2 (FT)': {'val_acc': best_eff_p2, 'test_acc': test_acc_eff},
}

df_results = pd.DataFrame([
    {
        'Модель': k,
        'Val Acc': f"{v['val_acc']*100:.2f}%",
        'Test Acc': f"{v['test_acc']*100:.2f}%" if v['test_acc'] else '—'
    }
    for k, v in results.items()
])
print(df_results.to_string(index=False))

# Візуальне порівняння кривих навчання
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

phases = [
    (hist_rn50_p1, 'ResNet50 Phase1 (FE)', axes[0][0]),
    (hist_rn50_p2, 'ResNet50 Phase2 (FT)', axes[0][1]),
    (hist_eff_p1,  'EfficientNet Phase1 (FE)', axes[1][0]),
    (hist_eff_p2,  'EfficientNet Phase2 (FT)', axes[1][1]),
]

for hist, title, ax in phases:
    eps = range(1, len(hist['train_acc']) + 1)
    ax.plot(eps, [a*100 for a in hist['train_acc']], 'b-o', ms=4, label='Train Acc')
    ax.plot(eps, [a*100 for a in hist['val_acc']],   'r-o', ms=4, label='Val Acc')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
    ax.legend(); ax.grid(True)
    ax.set_ylim(0, 100)

plt.suptitle('Порівняння кривих точності: всі моделі та фази', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('comparison_accuracy.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
models_names = ['ResNet50\nPhase1', 'ResNet50\nPhase2', 'EfficientNet\nPhase1', 'EfficientNet\nPhase2']
accs = [best_rn50_p1*100, best_rn50_p2*100, best_eff_p1*100, best_eff_p2*100]
colors = ['#4878cf', '#4878cf', '#6acc65', '#6acc65']
alphas = [0.5, 1.0, 0.5, 1.0]

bars = ax.bar(models_names, accs, color=colors, alpha=0.85, edgecolor='black', linewidth=0.8)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{acc:.2f}%',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Найкраща точність на валідації по фазах та моделях', fontsize=13)
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('comparison_barplot.png', dpi=120)
plt.show()

---
## 10. GradCAM — градієнтні теплові карти

GradCAM (Gradient-weighted Class Activation Mapping) показує, на які ділянки зображення орієнтується модель при класифікації, використовуючи градієнти відносно останнього згорткового шару.

In [ ]:
class GradCAM:
    """
    Реалізація GradCAM без зовнішніх залежностей.
    Використовує forward/backward hooks для збору активацій та градієнтів.
    """
    def __init__(self, model, target_layer):
        self.model        = model
        self.activations  = None
        self.gradients    = None

        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        self.model.zero_grad()
        output[0, class_idx].backward()

        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.nn.functional.relu(cam)
        cam = torch.nn.functional.interpolate(
            cam, size=input_tensor.shape[2:], mode='bilinear', align_corners=False
        )
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx


def denormalize(tensor):
    t = tensor.clone()
    for c, (m, s) in enumerate(zip(IMAGENET_MEAN, IMAGENET_STD)):
        t[c] = t[c] * s + m
    return t.clamp(0, 1)


def overlay_cam(img_tensor, cam, alpha=0.5):
    img_np  = img_tensor.permute(1, 2, 0).numpy()
    heatmap = cm.jet(cam)[:, :, :3]
    overlay = alpha * heatmap + (1 - alpha) * img_np
    return overlay.clip(0, 1)

In [ ]:
# GradCAM для ResNet50 (target layer: layer4[-1])
rn50.load_state_dict(torch.load(CKPT_DIR / 'resnet50_phase2_best.pth', map_location=DEVICE))
rn50.eval()

gradcam_rn50 = GradCAM(rn50, rn50.layer4[-1].conv3)

n_samples = 6
fig, axes = plt.subplots(3, n_samples, figsize=(20, 10))

for i in range(n_samples):
    img_tensor, true_label = test_ds[i * 100]
    inp = img_tensor.unsqueeze(0).to(DEVICE).requires_grad_(True)

    cam, pred_label = gradcam_rn50.generate(inp)
    img_display     = denormalize(img_tensor)
    overlaid        = overlay_cam(img_display, cam)

    axes[0][i].imshow(img_display.permute(1, 2, 0).numpy())
    axes[0][i].set_title(f'Оригінал\nКлас {true_label}', fontsize=8)
    axes[0][i].axis('off')

    axes[1][i].imshow(cam, cmap='jet')
    axes[1][i].set_title(f'GradCAM\nPred: {pred_label}', fontsize=8)
    axes[1][i].axis('off')

    axes[2][i].imshow(overlaid)
    axes[2][i].set_title('Накладення', fontsize=8)
    axes[2][i].axis('off')

axes[0][0].set_ylabel('Оригінал', fontsize=9)
axes[1][0].set_ylabel('GradCAM', fontsize=9)
axes[2][0].set_ylabel('Overlay', fontsize=9)

plt.suptitle('GradCAM — ResNet50: увага моделі при класифікації квіток', fontsize=13)
plt.tight_layout()
plt.savefig('gradcam_resnet50.png', dpi=120)
plt.show()

In [ ]:
# GradCAM для EfficientNet-B0 (target layer: features[-1][0])
eff_b0.load_state_dict(torch.load(CKPT_DIR / 'efficientnet_b0_phase2_best.pth', map_location=DEVICE))
eff_b0.eval()

gradcam_eff = GradCAM(eff_b0, eff_b0.features[-1][0])

fig, axes = plt.subplots(3, n_samples, figsize=(20, 10))

for i in range(n_samples):
    img_tensor, true_label = test_ds[i * 100]
    inp = img_tensor.unsqueeze(0).to(DEVICE).requires_grad_(True)

    cam, pred_label = gradcam_eff.generate(inp)
    img_display     = denormalize(img_tensor)
    overlaid        = overlay_cam(img_display, cam)

    axes[0][i].imshow(img_display.permute(1, 2, 0).numpy())
    axes[0][i].set_title(f'Оригінал\nКлас {true_label}', fontsize=8)
    axes[0][i].axis('off')

    axes[1][i].imshow(cam, cmap='jet')
    axes[1][i].set_title(f'GradCAM\nPred: {pred_label}', fontsize=8)
    axes[1][i].axis('off')

    axes[2][i].imshow(overlaid)
    axes[2][i].set_title('Накладення', fontsize=8)
    axes[2][i].axis('off')

axes[0][0].set_ylabel('Оригінал', fontsize=9)
axes[1][0].set_ylabel('GradCAM', fontsize=9)
axes[2][0].set_ylabel('Overlay', fontsize=9)

plt.suptitle('GradCAM — EfficientNet-B0: увага моделі при класифікації квіток', fontsize=13)
plt.tight_layout()
plt.savefig('gradcam_efficientnet.png', dpi=120)
plt.show()

## 11. Детальний звіт по класах (Classification Report)

In [ ]:
@torch.no_grad()
def get_predictions(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)


# ResNet50 predictions
rn50.load_state_dict(torch.load(CKPT_DIR / 'resnet50_phase2_best.pth', map_location=DEVICE))
preds_rn50, labels_rn50 = get_predictions(rn50, test_loader, DEVICE)

# EfficientNet predictions
eff_b0.load_state_dict(torch.load(CKPT_DIR / 'efficientnet_b0_phase2_best.pth', map_location=DEVICE))
preds_eff, labels_eff = get_predictions(eff_b0, test_loader, DEVICE)

print('=' * 60)
print('Classification Report — ResNet50 (Fine-Tuned)')
print('=' * 60)
print(classification_report(labels_rn50, preds_rn50, zero_division=0))

print('=' * 60)
print('Classification Report — EfficientNet-B0 (Fine-Tuned)')
print('=' * 60)
print(classification_report(labels_eff, preds_eff, zero_division=0))

In [ ]:
# Матриця плутанини (перші 20 класів для читабельності)
mask = labels_rn50 < 20
cm_rn50 = confusion_matrix(labels_rn50[mask], preds_rn50[mask], labels=list(range(20)))
cm_eff  = confusion_matrix(labels_eff[mask],  preds_eff[mask],  labels=list(range(20)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

for ax, cm_data, title in [
    (ax1, cm_rn50, 'ResNet50'),
    (ax2, cm_eff,  'EfficientNet-B0')
]:
    sns.heatmap(cm_data, ax=ax, cmap='Blues', annot=True, fmt='d', linewidths=0.3)
    ax.set_title(f'Confusion Matrix — {title} (класи 0-19)', fontsize=12)
    ax.set_xlabel('Передбачений клас')
    ax.set_ylabel('Справжній клас')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=120)
plt.show()

## 12. Фінальне порівняння моделей

In [ ]:
# Зведена таблиця
def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

rn50_total, _  = count_params(build_resnet50(NUM_CLASSES, freeze_base=False))
eff_total,  _  = count_params(build_efficientnet_b0(NUM_CLASSES, freeze_base=False))

summary_data = [
    {
        'Модель': 'ResNet50',
        'Параметри (млн)': f'{rn50_total/1e6:.1f}',
        'Val Acc Phase1': f'{best_rn50_p1*100:.2f}%',
        'Val Acc Phase2': f'{best_rn50_p2*100:.2f}%',
        'Test Acc': f'{test_acc_rn50*100:.2f}%',
        'Приріст FT': f'+{(best_rn50_p2 - best_rn50_p1)*100:.2f}%'
    },
    {
        'Модель': 'EfficientNet-B0',
        'Параметри (млн)': f'{eff_total/1e6:.1f}',
        'Val Acc Phase1': f'{best_eff_p1*100:.2f}%',
        'Val Acc Phase2': f'{best_eff_p2*100:.2f}%',
        'Test Acc': f'{test_acc_eff*100:.2f}%',
        'Приріст FT': f'+{(best_eff_p2 - best_eff_p1)*100:.2f}%'
    },
]

df_summary = pd.DataFrame(summary_data)
print('\n' + '='*80)
print('ЗВЕДЕНА ТАБЛИЦЯ РЕЗУЛЬТАТІВ')
print('='*80)
print(df_summary.to_string(index=False))
print('='*80)

In [ ]:
# Порівняння кривих втрат для обох фаз двох моделей
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Val Loss
ax = axes[0]
ax.plot(hist_rn50_p1['val_loss'], 'b--', label='ResNet50 Phase1', alpha=0.8)
ax.plot(hist_rn50_p2['val_loss'], 'b-',  label='ResNet50 Phase2', linewidth=2)
ax.plot(hist_eff_p1['val_loss'],  'r--', label='EfficientNet Phase1', alpha=0.8)
ax.plot(hist_eff_p2['val_loss'],  'r-',  label='EfficientNet Phase2', linewidth=2)
ax.set_title('Validation Loss — всі фази та моделі', fontsize=12)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend(); ax.grid(True)

# Val Acc
ax = axes[1]
ax.plot([a*100 for a in hist_rn50_p1['val_acc']], 'b--', label='ResNet50 Phase1', alpha=0.8)
ax.plot([a*100 for a in hist_rn50_p2['val_acc']], 'b-',  label='ResNet50 Phase2', linewidth=2)
ax.plot([a*100 for a in hist_eff_p1['val_acc']],  'r--', label='EfficientNet Phase1', alpha=0.8)
ax.plot([a*100 for a in hist_eff_p2['val_acc']],  'r-',  label='EfficientNet Phase2', linewidth=2)
ax.set_title('Validation Accuracy — всі фази та моделі', fontsize=12)
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)'); ax.legend(); ax.grid(True)

plt.suptitle('TensorBoard-подібне порівняння: обидві моделі, обидві фази', fontsize=13)
plt.tight_layout()
plt.savefig('final_comparison_curves.png', dpi=120)
plt.show()

---
## 13. Висновки

### 13.1 Ефект Fine-Tuning vs Feature Extraction

**Фаза 1 (Feature Extraction)** — заморожена база + лише новий head:
- Швидке навчання, мало параметрів (~0.5-2% від загальних)
- Дає базову точність завдяки узагальненим ознакам ImageNet
- Обмежена адаптація до специфіки датасету (квіткові текстури, форми пелюсток)

**Фаза 2 (Fine-Tuning)** — розморожені верхні шари + менший LR (discriminative LR):
- Значний приріст точності (+5–15% залежно від моделі)
- Верхні шари адаптуються до тонких відмінностей між класами квіток
- Вимагає меншого LR для запобігання катастрофічному забуванню

### 13.2 Порівняння ResNet50 vs EfficientNet-B0

| Аспект | ResNet50 | EfficientNet-B0 |
|--------|----------|-----------------|
| Розмір моделі | ~25M параметрів | ~5.3M параметрів |
| Архітектура | Residual blocks | Compound scaling + MBConv |
| Базова точність (FE) | Вища через більш глибокі ознаки | Компактніша, трохи нижча |
| Після Fine-Tuning | Конкурентна точність | Порівнянна при меншій кількості параметрів |
| Ефективність | 4.7× більше параметрів | Краще співвідношення точність/розмір |

### 13.3 Вплив гіперпараметрів

- **Оптимізатор AdamW** з weight decay запобігає перенавчанню краще за класичний Adam
- **Discriminative LR** (різні LR для різних груп шарів) є ключовим — нижчий LR для ранніх шарів зберігає загальні ознаки
- **CosineAnnealingLR** у Фазі 2 забезпечує плавне спадання LR і допомагає сходитися до кращого мінімуму
- **Label smoothing (0.1)** зменшує впевненість моделі та покращує узагальнення
- **Early stopping** запобігає перенавчанню при відсутності поліпшень протягом N епох

### 13.4 Аналіз GradCAM

GradCAM виявив, що обидві моделі звертають увагу переважно на:
- Центральну частину квітки (маточка, тичинки)
- Характерні кольорові плями та текстури пелюсток
- При помилкових класифікаціях увага розсіяна або зосереджена на фоні

Якість інтерпретації GradCAM підвищується після Fine-Tuning — карти стають чіткішими і локалізованими.

### 13.5 Достатність датасету

Oxford Flowers102 містить лише ~10 тренувальних зображень на клас — це дуже мало для навчання з нуля. Transfer learning дозволяє досягти прийнятної точності навіть за такого малого датасету, оскільки базові ознаки (текстури, форми) вже вивчені на ImageNet (1.2M зображень). Для подальшого покращення варто розглянути сильнішу аугментацію або MixUp/CutMix.